# Notebook 01 — Reproducible interval preprocessing

This notebook preprocesses the canonical merged CSVs created by Notebook 00.
It does not merge raw files. It preserves high values, separates LP from DBE,
audits timestamps and duplicates, applies a past-valid network-metadata join,
and creates the reusable LP interval CSV plus supporting audit CSVs.

Missing interval demand is flagged and retained. No interval-level imputation,
outlier deletion, clipping, or winsorization is performed.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import gc
import hashlib
import json
import os
import platform
import random
import shutil
import sys

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
if not (PROJECT_ROOT / "data").is_dir() or not (PROJECT_ROOT / "notebooks").is_dir():
    raise RuntimeError(f"Could not identify project root: {PROJECT_ROOT}")

INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_TABLE_DIR = PROJECT_ROOT / "reports" / "tables"

RAW_MERGED_DATA_PATH = INTERIM_DATA_DIR / "01_merged_interval_data.csv"
SIX_HOUR_MERGED_DATA_PATH = INTERIM_DATA_DIR / "01_merged_6hour_interval_data.csv"
MERGE_MANIFEST_PATH = INTERIM_DATA_DIR / "00_merge_manifest.json"

CSV_ENCODING = "utf-8"
CSV_SEPARATOR = ","
CSV_MISSING_VALUE = ""
CHUNK_SIZE = 250_000
SIX_HOUR_CHUNK_SIZE = 250_000
DUPLICATE_PARTITION_COUNT = 64
EXECUTION_MODE = "rebuild"

for directory in [INTERIM_DATA_DIR, PROCESSED_DATA_DIR, REPORT_TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

environment_table = pd.DataFrame(
    {
        "item": [
            "executed_at_utc", "python_executable", "python_version",
            "platform", "pandas_version", "numpy_version", "seed",
            "project_root", "execution_mode", "chunk_size",
            "duplicate_partitions",
        ],
        "value": [
            datetime.now(timezone.utc).isoformat(), sys.executable,
            platform.python_version(), platform.platform(), pd.__version__,
            np.__version__, SEED, str(PROJECT_ROOT), EXECUTION_MODE,
            CHUNK_SIZE, DUPLICATE_PARTITION_COUNT,
        ],
    }
)
display(environment_table)

## 1. CSV input contract

Notebook 00 owns the merge. This notebook fails if either canonical CSV or the
successful merge manifest is absent. CSV fields are read using explicit types;
numeric and datetime interpretation happens only inside the documented
normalization function.

In [ ]:
if not RAW_MERGED_DATA_PATH.exists():
    raise FileNotFoundError(f"Merged interval CSV is missing: {RAW_MERGED_DATA_PATH}")
if not SIX_HOUR_MERGED_DATA_PATH.exists():
    raise FileNotFoundError(f"Merged six-hour CSV is missing: {SIX_HOUR_MERGED_DATA_PATH}")
if not MERGE_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Notebook 00 manifest is missing: {MERGE_MANIFEST_PATH}")

merge_manifest = json.loads(MERGE_MANIFEST_PATH.read_text(encoding="utf-8"))
if not merge_manifest.get("all_validations_passed", False):
    raise RuntimeError("Notebook 00 merge manifest did not pass all validation gates")

CANONICAL_INPUT_COLUMNS = [
    "household_id", "timestamp", "date", "time", "obis",
    "import_kwh", "export_kwh", "import_kvarh", "export_kvarh",
    "power_factor", "avg_current", "avg_export_kw", "avg_import_kw",
    "avg_voltage", "phase_a_inst_current", "phase_a_inst_voltage",
    "phase_b_inst_current", "phase_b_inst_voltage",
    "phase_c_inst_current", "phase_c_inst_voltage",
    "source_file", "source_row_number",
]

NUMERIC_COLUMNS = [
    "import_kwh", "export_kwh", "import_kvarh", "export_kvarh",
    "power_factor", "avg_current", "avg_export_kw", "avg_import_kw",
    "avg_voltage", "phase_a_inst_current", "phase_a_inst_voltage",
    "phase_b_inst_current", "phase_b_inst_voltage",
    "phase_c_inst_current", "phase_c_inst_voltage",
]

RAW_MERGED_DTYPE_MAP = {column: "string" for column in CANONICAL_INPUT_COLUMNS}

def read_dataset_csv(path, dtype_map=None, date_columns=None, chunksize=None, usecols=None):
    return pd.read_csv(
        path,
        sep=CSV_SEPARATOR,
        dtype=dtype_map,
        parse_dates=date_columns,
        true_values=["True"],
        false_values=["False"],
        keep_default_na=False,
        na_values=[CSV_MISSING_VALUE],
        encoding=CSV_ENCODING,
        chunksize=chunksize,
        usecols=usecols,
        low_memory=False,
    )

def iter_raw_chunks(columns=None, batch_size=CHUNK_SIZE):
    dtype_map = {
        column: RAW_MERGED_DTYPE_MAP[column]
        for column in (columns or CANONICAL_INPUT_COLUMNS)
    }
    yield from read_dataset_csv(
        RAW_MERGED_DATA_PATH,
        dtype_map=dtype_map,
        chunksize=batch_size,
        usecols=columns,
    )

interval_header = list(pd.read_csv(RAW_MERGED_DATA_PATH, nrows=0).columns)
six_hour_header = list(pd.read_csv(SIX_HOUR_MERGED_DATA_PATH, nrows=0).columns)
input_contract_checks = pd.DataFrame(
    [
        ("merged_interval_is_csv", RAW_MERGED_DATA_PATH.suffix.lower() == ".csv", str(RAW_MERGED_DATA_PATH)),
        ("merged_six_hour_is_csv", SIX_HOUR_MERGED_DATA_PATH.suffix.lower() == ".csv", str(SIX_HOUR_MERGED_DATA_PATH)),
        ("interval_header_matches", interval_header == CANONICAL_INPUT_COLUMNS, str(interval_header)),
        ("six_hour_required_columns", not set(["household_id", "date", "time", "branch", "depot", "substation_id"]) - set(six_hour_header), str(six_hour_header)),
        ("merge_manifest_valid", merge_manifest.get("all_validations_passed") is True, str(merge_manifest.get("all_validations_passed"))),
    ],
    columns=["check", "passed", "evidence"],
)
display(input_contract_checks)
if not input_contract_checks["passed"].all():
    raise RuntimeError("CSV input contract failed")

## 2. Evidence before scientific configuration

The complete merged interval CSV is scanned before the values 96 and 70 are
configured. This produces the evidence used to justify a 15-minute cadence,
96 expected intervals per day, and the descriptive near-complete boundary.

In [ ]:
discovery_total_rows = 0
discovery_households = set()
discovery_obis_counts = Counter()
discovery_source_counts = Counter()
discovery_time_counts = Counter()
discovery_timestamp_difference_histogram = Counter()
preliminary_observed_counts = Counter()
preliminary_valid_demand_counts = Counter()
discovery_lp_missing_import = 0
discovery_lp_negative_import = 0
discovery_lp_max_import = float("-inf")
discovery_min_date = None
discovery_max_date = None
extreme_sample_parts = []

discovery_columns = [
    "household_id", "timestamp", "date", "time", "obis",
    "avg_import_kw", "source_file",
]

for chunk_number, chunk in enumerate(iter_raw_chunks(columns=discovery_columns)):
    discovery_total_rows += len(chunk)
    household = chunk["household_id"].astype("string").str.strip()
    obis = chunk["obis"].astype("string").str.strip().str.upper()
    date_text = chunk["date"].astype("string").str.strip()
    time_text = chunk["time"].astype("string").str.strip()
    import_kw = pd.to_numeric(chunk["avg_import_kw"], errors="coerce")

    discovery_households.update(household.dropna().unique().tolist())
    discovery_obis_counts.update(obis.fillna("<MISSING>").astype(str))
    discovery_source_counts.update(
        chunk["source_file"].astype("string").fillna("<MISSING>").astype(str)
    )

    valid_dates = pd.to_datetime(date_text, errors="coerce")
    if valid_dates.notna().any():
        chunk_min_date = valid_dates.min()
        chunk_max_date = valid_dates.max()
        discovery_min_date = chunk_min_date if discovery_min_date is None else min(discovery_min_date, chunk_min_date)
        discovery_max_date = chunk_max_date if discovery_max_date is None else max(discovery_max_date, chunk_max_date)

    lp_mask = obis.eq("LP").fillna(False)
    lp_household = household.loc[lp_mask]
    lp_date = date_text.loc[lp_mask]
    lp_time = time_text.loc[lp_mask]
    lp_import = import_kw.loc[lp_mask]

    discovery_time_counts.update(lp_time.dropna().astype(str))
    discovery_lp_missing_import += int(lp_import.isna().sum())
    discovery_lp_negative_import += int(lp_import.lt(0).fillna(False).sum())
    if lp_import.notna().any():
        discovery_lp_max_import = max(discovery_lp_max_import, float(lp_import.max()))

    observed_keys = pd.DataFrame({"household_id": lp_household, "date": lp_date}).dropna()
    preliminary_observed_counts.update(
        observed_keys.groupby(["household_id", "date"], sort=False).size().to_dict()
    )
    valid_demand_mask = lp_import.notna() & lp_import.ge(0)
    valid_keys = pd.DataFrame(
        {
            "household_id": lp_household.loc[valid_demand_mask],
            "date": lp_date.loc[valid_demand_mask],
        }
    ).dropna()
    preliminary_valid_demand_counts.update(
        valid_keys.groupby(["household_id", "date"], sort=False).size().to_dict()
    )

    local_naive = pd.to_datetime(date_text + " " + time_text, errors="coerce")
    numeric_timestamp = pd.to_numeric(chunk["timestamp"], errors="coerce")
    timestamp_utc = pd.to_datetime(numeric_timestamp, unit="ms", utc=True, errors="coerce")
    numeric_local = timestamp_utc.dt.tz_convert("Asia/Colombo").dt.tz_localize(None)
    rounded_difference = (
        (local_naive - numeric_local).dt.total_seconds().div(60).dropna().round(1)
    )
    discovery_timestamp_difference_histogram.update(
        rounded_difference.value_counts().to_dict()
    )

    sample_values = lp_import.iloc[::50].dropna()
    if not sample_values.empty:
        extreme_sample_parts.append(sample_values.astype("float64"))

    if (chunk_number + 1) % 40 == 0:
        print(f"Evidence scan: {discovery_total_rows:,} rows")

print(f"Evidence scan completed: {discovery_total_rows:,} rows")

In [ ]:
def weighted_quantile_from_counter(counter, quantile):
    if not counter:
        return np.nan
    ordered = sorted((float(value), int(count)) for value, count in counter.items())
    total = sum(count for _, count in ordered)
    target = quantile * (total - 1)
    cumulative = 0
    for value, count in ordered:
        cumulative += count
        if cumulative - 1 >= target:
            return value
    return ordered[-1][0]

parsed_times = pd.to_datetime(
    pd.Series(sorted(discovery_time_counts), dtype="string"),
    format="%H:%M:%S",
    errors="coerce",
)
unique_slot_minutes = sorted(
    (parsed_times.dt.hour * 60 + parsed_times.dt.minute)
    .dropna().astype(int).unique().tolist()
)
if len(unique_slot_minutes) < 2:
    raise ValueError("Insufficient observed time values to derive cadence")

circular_differences = np.diff(unique_slot_minutes + [unique_slot_minutes[0] + 1440])
positive_differences = circular_differences[circular_differences > 0]
observed_interval_minutes = int(pd.Series(positive_differences).mode().iloc[0])
derived_expected_intervals_per_day = int(1440 / observed_interval_minutes)

preliminary_day_keys = sorted(
    set(preliminary_observed_counts) | set(preliminary_valid_demand_counts)
)
preliminary_coverage_table = pd.DataFrame(
    {
        "household_id": [key[0] for key in preliminary_day_keys],
        "date": [key[1] for key in preliminary_day_keys],
        "observed_lp_rows": [preliminary_observed_counts.get(key, 0) for key in preliminary_day_keys],
        "valid_demand_rows": [preliminary_valid_demand_counts.get(key, 0) for key in preliminary_day_keys],
    }
)

DESCRIPTIVE_NEAR_COMPLETE_MIN_INTERVALS = 70
preliminary_coverage_table["observed_row_category"] = np.select(
    [
        preliminary_coverage_table["observed_lp_rows"].eq(derived_expected_intervals_per_day),
        preliminary_coverage_table["observed_lp_rows"].ge(DESCRIPTIVE_NEAR_COMPLETE_MIN_INTERVALS),
    ],
    ["complete", "near_complete"],
    default="severely_incomplete",
)
preliminary_coverage_table["valid_demand_category"] = np.select(
    [
        preliminary_coverage_table["valid_demand_rows"].eq(derived_expected_intervals_per_day),
        preliminary_coverage_table["valid_demand_rows"].ge(DESCRIPTIVE_NEAR_COMPLETE_MIN_INTERVALS),
    ],
    ["complete", "near_complete"],
    default="severely_incomplete",
)

cadence_evidence = pd.DataFrame(
    {
        "measure": [
            "unique_observed_time_slots", "modal_interval_minutes",
            "derived_expected_intervals_per_day",
            "descriptive_near_complete_min_intervals", "preliminary_household_days",
        ],
        "value": [
            len(unique_slot_minutes), observed_interval_minutes,
            derived_expected_intervals_per_day,
            DESCRIPTIVE_NEAR_COMPLETE_MIN_INTERVALS,
            len(preliminary_coverage_table),
        ],
    }
)
preliminary_coverage_summary = pd.concat(
    [
        preliminary_coverage_table["observed_row_category"].value_counts().rename("household_days").to_frame().assign(measure="observed_lp_rows"),
        preliminary_coverage_table["valid_demand_category"].value_counts().rename("household_days").to_frame().assign(measure="valid_nonmissing_demand_rows"),
    ]
).rename_axis("coverage_category").reset_index()
timestamp_alignment_evidence = pd.DataFrame(
    {
        "measure": [
            "difference_minutes_p00", "difference_minutes_p25",
            "difference_minutes_p50", "difference_minutes_p75",
            "difference_minutes_p100",
        ],
        "value": [
            weighted_quantile_from_counter(discovery_timestamp_difference_histogram, q)
            for q in [0.00, 0.25, 0.50, 0.75, 1.00]
        ],
    }
)

display(cadence_evidence)
display(preliminary_coverage_summary)
display(preliminary_coverage_table[["observed_lp_rows", "valid_demand_rows"]].describe().T)
display(timestamp_alignment_evidence)

## 3. Scientific policy configuration

The configuration below is intentionally placed after the cadence and coverage
evidence. A day is near-complete at 70–95 valid intervals for descriptive audit.
Coverage flags summarize observed quality; target-day and history eligibility are
selected explicitly in downstream target-design/modeling stages.

In [ ]:
TIMEZONE = "Asia/Colombo"
PRIMARY_OBIS = "LP"
EXPECTED_INTERVALS_PER_DAY = int(derived_expected_intervals_per_day)
DESCRIPTIVE_NEAR_COMPLETE_MIN_INTERVALS = 70
CANONICAL_TIMESTAMP_SOURCE = "date_plus_time_local_wall_clock"
RETAIN_STATISTICAL_EXTREMES = True
REMOVE_EXACT_DUPLICATES = True
QUARANTINE_CONFLICTING_DUPLICATES = True
ALLOW_FUTURE_METADATA_FILL = False

if observed_interval_minutes != 15 or EXPECTED_INTERVALS_PER_DAY != 96:
    raise ValueError(
        f"Observed cadence was {observed_interval_minutes} minutes and implied "
        f"{EXPECTED_INTERVALS_PER_DAY} intervals/day"
    )
if not 0 < DESCRIPTIVE_NEAR_COMPLETE_MIN_INTERVALS < EXPECTED_INTERVALS_PER_DAY:
    raise ValueError("Descriptive near-complete boundary must be below the observed-day interval count")

policy_decisions = pd.DataFrame(
    {
        "policy": [
            "timezone", "primary_obis", "expected_intervals_per_day",
            "descriptive_near_complete_min_intervals", "day_completeness_role",
            "canonical_timestamp_source", "retain_statistical_extremes",
            "allow_future_metadata_fill", "interval_imputation",
        ],
        "value": [
            TIMEZONE, PRIMARY_OBIS, EXPECTED_INTERVALS_PER_DAY,
            DESCRIPTIVE_NEAR_COMPLETE_MIN_INTERVALS, "descriptive_only; target/history eligibility selected downstream",
            CANONICAL_TIMESTAMP_SOURCE, RETAIN_STATISTICAL_EXTREMES,
            ALLOW_FUTURE_METADATA_FILL, "not_applied",
        ],
        "evidence_or_reason": [
            "Sri Lankan local calendar-day analysis",
            "LP is the regular load-profile population",
            f"Observed {observed_interval_minutes}-minute cadence: 1440/{observed_interval_minutes}",
            "Research descriptive boundary evaluated above",
            "An absent slot could contain the true daily peak",
            "Date and time define the regular 15-minute local grid",
            "High demand is the research phenomenon",
            "Future metadata would leak backward in time",
            "Missing intervals remain explicitly missing",
        ],
    }
)
display(policy_decisions)

extreme_sample = pd.concat(extreme_sample_parts, ignore_index=True)
q1_import = float(extreme_sample.quantile(0.25))
q3_import = float(extreme_sample.quantile(0.75))
global_iqr_extreme_limit = q3_import + 1.5 * (q3_import - q1_import)
global_top_1pct_limit = float(extreme_sample.quantile(0.99))
global_top_0_1pct_limit = float(extreme_sample.quantile(0.999))
extreme_reference_table = pd.DataFrame(
    {
        "measure": ["sample_rows", "q1", "q3", "iqr_upper", "p99", "p99_9", "observed_lp_max"],
        "value": [
            len(extreme_sample), q1_import, q3_import,
            global_iqr_extreme_limit, global_top_1pct_limit,
            global_top_0_1pct_limit, discovery_lp_max_import,
        ],
        "use": [
            "deterministic audit sample", "EDA only", "EDA only",
            "EDA flag only", "EDA flag only", "EDA flag only",
            "retained measurement",
        ],
    }
)
display(extreme_reference_table)

## 4. Time-valid six-hour network metadata

Six-hour metadata is converted to change events. Interval rows receive only the
most recent event at or before their local measurement time. No future event is
carried backward. Rows before a household's first valid event remain `Unknown`
with the explicit status `unknown_before_first_metadata`.

In [ ]:
NETWORK_METADATA_COLUMNS = ["branch", "depot", "substation_id"]
metadata_frames = []
metadata_invalid_timestamp_count = 0

metadata_dtype = {
    column: "string"
    for column in ["household_id", "date", "time", *NETWORK_METADATA_COLUMNS]
}
for chunk in read_dataset_csv(
    SIX_HOUR_MERGED_DATA_PATH,
    dtype_map=metadata_dtype,
    chunksize=SIX_HOUR_CHUNK_SIZE,
    usecols=list(metadata_dtype),
):
    for column in ["household_id", *NETWORK_METADATA_COLUMNS]:
        chunk[column] = chunk[column].astype("string").str.strip()
    chunk["metadata_valid_from"] = pd.to_datetime(
        chunk["date"].astype("string") + " " + chunk["time"].astype("string"),
        errors="coerce",
    )
    metadata_invalid_timestamp_count += int(chunk["metadata_valid_from"].isna().sum())
    metadata_frames.append(
        chunk.dropna(subset=["household_id", "metadata_valid_from"])[
            ["household_id", "metadata_valid_from", *NETWORK_METADATA_COLUMNS]
        ]
    )

metadata = pd.concat(metadata_frames, ignore_index=True)
metadata = metadata.drop_duplicates().sort_values(
    ["household_id", "metadata_valid_from"], kind="mergesort"
)

same_time_cardinality = metadata.groupby(
    ["household_id", "metadata_valid_from"], sort=False
)[NETWORK_METADATA_COLUMNS].nunique(dropna=False)
conflicting_metadata_keys = same_time_cardinality.gt(1).any(axis=1)
metadata_same_time_conflict_count = int(conflicting_metadata_keys.sum())
if metadata_same_time_conflict_count:
    raise ValueError(
        f"Conflicting network metadata at the same household/time: {metadata_same_time_conflict_count} keys"
    )

metadata = metadata.drop_duplicates(
    ["household_id", "metadata_valid_from"], keep="last"
).reset_index(drop=True)
metadata_cardinality = metadata.groupby("household_id", sort=False)[
    NETWORK_METADATA_COLUMNS
].nunique(dropna=False)
changing_metadata_households = set(
    metadata_cardinality.index[metadata_cardinality.gt(1).any(axis=1)].astype(str)
)

previous_metadata = metadata.groupby("household_id", sort=False)[
    NETWORK_METADATA_COLUMNS
].shift()
first_event = metadata["household_id"].ne(metadata["household_id"].shift())
value_changed = metadata[NETWORK_METADATA_COLUMNS].fillna("<MISSING>").ne(
    previous_metadata.fillna("<MISSING>")
).any(axis=1)
metadata_events = metadata.loc[
    first_event | value_changed,
    ["household_id", "metadata_valid_from", *NETWORK_METADATA_COLUMNS],
].copy()
metadata_events = metadata_events.sort_values(
    ["metadata_valid_from", "household_id"], kind="mergesort"
).reset_index(drop=True)

changing_metadata_detail = metadata[
    metadata["household_id"].astype(str).isin(changing_metadata_households)
][["household_id", "metadata_valid_from", *NETWORK_METADATA_COLUMNS]].copy()

network_metadata_audit = pd.DataFrame(
    {
        "measure": [
            "valid_six_hour_rows", "invalid_metadata_timestamps",
            "metadata_households", "metadata_change_events",
            "changing_metadata_households", "same_time_conflicting_keys",
        ],
        "value": [
            len(metadata), metadata_invalid_timestamp_count,
            metadata["household_id"].nunique(), len(metadata_events),
            len(changing_metadata_households), metadata_same_time_conflict_count,
        ],
    }
)
display(network_metadata_audit)
display(changing_metadata_detail.head(30))

del metadata_frames, metadata, metadata_cardinality, previous_metadata
gc.collect()

## 5. Output contract and transformation helpers

The primary CSV contains the fields required by interval EDA and household-day
construction. Measurements not selected for the analytical output remain intact
in Notebook 00's lossless merged CSV. Temporary duplicate partitions use
contiguous household ranges, so processing partitions in numeric order produces
a final file globally ordered by household and local timestamp.

In [ ]:
PRIMARY_OUTPUT_PATH = PROCESSED_DATA_DIR / "01_preprocessed_lp_interval_data.csv"
PRIMARY_TEMP_PATH = PROCESSED_DATA_DIR / "01_preprocessed_lp_interval_data.tmp.csv"
DBE_AUDIT_PATH = INTERIM_DATA_DIR / "01_dbe_records.csv"
INVALID_RECORDS_PATH = INTERIM_DATA_DIR / "01_invalid_key_records.csv"
DUPLICATE_CONFLICTS_PATH = INTERIM_DATA_DIR / "01_duplicate_conflicts.csv"
DAY_COVERAGE_PATH = INTERIM_DATA_DIR / "01_household_day_coverage_audit.csv"
MANIFEST_PATH = PROCESSED_DATA_DIR / "01_preprocessing_manifest.json"
DTYPE_CONTRACT_PATH = PROCESSED_DATA_DIR / "01_preprocessing_dtype_contract.json"
WORK_DIR = INTERIM_DATA_DIR / "_preprocessing_work"
PARTITION_DIR = WORK_DIR / "household_partitions"

PRIMARY_MEASUREMENT_COLUMNS = [
    "avg_import_kw", "avg_current", "avg_voltage", "power_factor",
    "phase_a_inst_current", "phase_b_inst_current", "phase_c_inst_current",
    "import_kwh", "export_kwh", "avg_export_kw",
]
BOOLEAN_OUTPUT_COLUMNS = [
    "is_valid_for_daily_aggregation", "is_missing_import_demand",
    "is_negative_import_demand", "is_zero_import", "has_invalid_numeric_text",
    "is_imputed", "is_global_iqr_extreme_eda_only",
    "is_global_top_1pct_eda_only", "is_global_top_0_1pct_eda_only",
    "is_off_15min_grid", "is_duplicate_conflict",
]
PRIMARY_OUTPUT_COLUMNS = [
    "household_id", "measurement_timestamp_local", "measurement_date",
    "measurement_time", "time_slot_index", "obis_normalized",
    *PRIMARY_MEASUREMENT_COLUMNS,
    "is_valid_for_daily_aggregation", "is_missing_import_demand",
    "is_negative_import_demand", "is_zero_import", "has_invalid_numeric_text",
    "is_imputed", "interval_exclusion_reason", "quality_status",
    "is_global_iqr_extreme_eda_only", "is_global_top_1pct_eda_only",
    "is_global_top_0_1pct_eda_only", "timestamp_difference_minutes",
    "timestamp_alignment_status", "network_metadata_match_type",
    "branch", "depot", "substation_id", "source_file", "source_row_number",
    "is_off_15min_grid", "duplicate_status", "is_duplicate_conflict",
]
STAGING_COLUMNS = [
    "household_id", "measurement_timestamp_local", "measurement_date",
    "measurement_time", "time_slot_index", "obis_normalized", *NUMERIC_COLUMNS,
    "has_invalid_numeric_text", "timestamp_difference_minutes",
    "timestamp_alignment_status", "is_off_15min_grid",
    "is_missing_import_demand", "is_negative_import_demand", "is_zero_import",
    "source_file", "source_row_number",
]

PRIMARY_DTYPE_MAP = {
    **{column: "float64" for column in PRIMARY_MEASUREMENT_COLUMNS},
    **{column: "boolean" for column in BOOLEAN_OUTPUT_COLUMNS},
    "household_id": "string", "measurement_time": "string",
    "obis_normalized": "string", "interval_exclusion_reason": "string",
    "quality_status": "string", "timestamp_alignment_status": "string",
    "network_metadata_match_type": "string", "branch": "string",
    "depot": "string", "substation_id": "string", "source_file": "string",
    "duplicate_status": "string", "time_slot_index": "Int16",
    "source_row_number": "Int64", "timestamp_difference_minutes": "float64",
}
PRIMARY_DATE_COLUMNS = ["measurement_timestamp_local", "measurement_date"]

def ensure_inside_project(path):
    resolved = path.resolve()
    if PROJECT_ROOT not in resolved.parents and resolved != PROJECT_ROOT:
        raise RuntimeError(f"Refusing filesystem operation outside project: {resolved}")
    return resolved

def temp_csv_path(path):
    return path.with_name(f"{path.stem}.tmp.csv")

def append_csv_chunk(frame, temporary_path, first_chunk):
    frame.to_csv(
        temporary_path,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
        encoding=CSV_ENCODING,
        na_rep=CSV_MISSING_VALUE,
        date_format="%Y-%m-%d %H:%M:%S",
        lineterminator=chr(10),
    )

def write_csv_atomic(frame, output_path):
    temporary_path = temp_csv_path(output_path)
    if temporary_path.exists():
        temporary_path.unlink()
    append_csv_chunk(frame, temporary_path, True)
    os.replace(temporary_path, output_path)

def write_json_atomic(payload, output_path):
    temporary_path = output_path.with_suffix(output_path.suffix + ".tmp")
    temporary_path.write_text(
        json.dumps(payload, indent=2, default=str), encoding="utf-8"
    )
    os.replace(temporary_path, output_path)

def sha256_file(path, block_size=16 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

def normalize_interval_chunk(chunk):
    chunk = chunk.copy()
    for column in ["household_id", "timestamp", "date", "time", "obis", "source_file"]:
        chunk[column] = chunk[column].astype("string").str.strip()
    chunk["source_row_number"] = pd.to_numeric(
        chunk["source_row_number"], errors="coerce"
    ).astype("Int64")
    chunk["obis_normalized"] = chunk["obis"].str.upper()

    invalid_numeric_masks = []
    for column in NUMERIC_COLUMNS:
        original = chunk[column]
        numeric = pd.to_numeric(original, errors="coerce")
        invalid_numeric_masks.append(
            original.notna() & original.astype("string").str.strip().ne("") & numeric.isna()
        )
        chunk[column] = numeric.astype("float64")
    chunk["has_invalid_numeric_text"] = pd.concat(
        invalid_numeric_masks, axis=1
    ).any(axis=1).astype(bool)

    local_naive = pd.to_datetime(
        chunk["date"] + " " + chunk["time"], errors="coerce"
    )
    chunk["measurement_timestamp_local"] = local_naive
    chunk["measurement_date"] = local_naive.dt.normalize()
    chunk["measurement_time"] = local_naive.dt.strftime("%H:%M:%S").astype("string")

    numeric_timestamp = pd.to_numeric(chunk["timestamp"], errors="coerce")
    timestamp_utc = pd.to_datetime(numeric_timestamp, unit="ms", utc=True, errors="coerce")
    numeric_local_naive = timestamp_utc.dt.tz_convert(TIMEZONE).dt.tz_localize(None)
    chunk["timestamp_difference_minutes"] = (
        local_naive - numeric_local_naive
    ).dt.total_seconds().div(60).astype("float64")
    absolute_difference = chunk["timestamp_difference_minutes"].abs()
    chunk["timestamp_alignment_status"] = np.select(
        [
            local_naive.isna().to_numpy(dtype=bool),
            timestamp_utc.isna().to_numpy(dtype=bool),
            absolute_difference.le(1).fillna(False).to_numpy(dtype=bool),
            absolute_difference.le(90).fillna(False).to_numpy(dtype=bool),
        ],
        [
            "invalid_date_time", "invalid_numeric_timestamp",
            "aligned_within_1_minute", "rounded_or_delayed_within_90_minutes",
        ],
        default="large_timestamp_mismatch",
    ).astype(str)

    on_grid = (
        local_naive.notna()
        & local_naive.dt.minute.isin([0, 15, 30, 45])
        & local_naive.dt.second.eq(0)
        & local_naive.dt.microsecond.eq(0)
    )
    chunk["is_off_15min_grid"] = (~on_grid).astype(bool)
    chunk["time_slot_index"] = (
        local_naive.dt.hour * 4 + local_naive.dt.minute // 15
    ).astype("Int16")
    chunk["is_missing_import_demand"] = chunk["avg_import_kw"].isna().astype(bool)
    chunk["is_negative_import_demand"] = chunk["avg_import_kw"].lt(0).fillna(False).astype(bool)
    chunk["is_zero_import"] = chunk["avg_import_kw"].eq(0).fillna(False).astype(bool)
    chunk["is_valid_interval_key"] = (
        chunk["household_id"].notna()
        & chunk["household_id"].ne("")
        & local_naive.notna()
        & on_grid
    ).astype(bool)
    return chunk

outputs_to_rebuild = [
    PRIMARY_OUTPUT_PATH, PRIMARY_TEMP_PATH, DBE_AUDIT_PATH,
    INVALID_RECORDS_PATH, DUPLICATE_CONFLICTS_PATH, DAY_COVERAGE_PATH,
    MANIFEST_PATH, DTYPE_CONTRACT_PATH,
]
if EXECUTION_MODE == "rebuild":
    if WORK_DIR.exists():
        ensure_inside_project(WORK_DIR)
        shutil.rmtree(WORK_DIR)
    for output_path in outputs_to_rebuild:
        if output_path.exists():
            ensure_inside_project(output_path)
            output_path.unlink()
else:
    raise ValueError("Notebook 01 currently supports reproducible rebuild mode only")
PARTITION_DIR.mkdir(parents=True, exist_ok=True)

sorted_households = sorted(discovery_households)
household_partition_lookup = {
    household_id: min(
        int(index * DUPLICATE_PARTITION_COUNT / len(sorted_households)),
        DUPLICATE_PARTITION_COUNT - 1,
    )
    for index, household_id in enumerate(sorted_households)
}

## 6. LP staging and DBE/invalid-record separation

Valid-key LP records are written to 64 household-range CSV partitions. All rows
for one household therefore enter the same partition, enabling an exact complete
duplicate audit without loading the full interval population into memory.

In [ ]:
partition_first_write = defaultdict(lambda: True)
dbe_temp_path = temp_csv_path(DBE_AUDIT_PATH)
invalid_temp_path = temp_csv_path(INVALID_RECORDS_PATH)
dbe_first_write = True
invalid_first_write = True
staging_counts = Counter()
staging_timestamp_status = Counter()
staging_invalid_numeric_rows = 0

for chunk_number, raw_chunk in enumerate(iter_raw_chunks()):
    processed = normalize_interval_chunk(raw_chunk)
    staging_counts["input_rows"] += len(processed)

    dbe_mask = processed["obis_normalized"].eq("DBE").fillna(False)
    lp_mask = processed["obis_normalized"].eq(PRIMARY_OBIS).fillna(False)
    unexpected_mask = ~(dbe_mask | lp_mask)
    staging_counts["dbe_rows"] += int(dbe_mask.sum())
    staging_counts["lp_rows"] += int(lp_mask.sum())
    staging_counts["unexpected_obis_rows"] += int(unexpected_mask.sum())
    staging_invalid_numeric_rows += int(processed["has_invalid_numeric_text"].sum())
    staging_timestamp_status.update(processed["timestamp_alignment_status"].astype(str))

    dbe_records = processed.loc[dbe_mask, STAGING_COLUMNS]
    if not dbe_records.empty:
        append_csv_chunk(dbe_records, dbe_temp_path, dbe_first_write)
        dbe_first_write = False

    invalid_mask = (lp_mask & ~processed["is_valid_interval_key"]) | unexpected_mask
    invalid_records = processed.loc[invalid_mask, STAGING_COLUMNS].copy()
    if not invalid_records.empty:
        invalid_records["record_exclusion_reason"] = np.where(
            unexpected_mask.loc[invalid_records.index].to_numpy(dtype=bool),
            "unexpected_obis",
            "invalid_household_or_timestamp_key",
        )
        append_csv_chunk(invalid_records, invalid_temp_path, invalid_first_write)
        invalid_first_write = False

    valid_lp = processed.loc[
        lp_mask & processed["is_valid_interval_key"], STAGING_COLUMNS
    ].copy()
    staging_counts["valid_key_lp_rows"] += len(valid_lp)
    staging_counts["invalid_key_lp_rows"] += int(
        (lp_mask & ~processed["is_valid_interval_key"]).sum()
    )

    if not valid_lp.empty:
        valid_lp["_duplicate_partition"] = valid_lp["household_id"].map(
            household_partition_lookup
        ).astype("Int16")
        if valid_lp["_duplicate_partition"].isna().any():
            raise RuntimeError("A valid LP household was absent from the partition lookup")
        for partition_id, partition_frame in valid_lp.groupby(
            "_duplicate_partition", sort=False
        ):
            partition_id = int(partition_id)
            partition_path = PARTITION_DIR / f"partition_{partition_id:02d}.csv"
            partition_frame = partition_frame.drop(columns="_duplicate_partition")
            append_csv_chunk(
                partition_frame,
                partition_path,
                partition_first_write[partition_id],
            )
            partition_first_write[partition_id] = False

    if (chunk_number + 1) % 25 == 0:
        print(f"Staged {staging_counts['input_rows']:,} rows")

if dbe_first_write:
    append_csv_chunk(pd.DataFrame(columns=STAGING_COLUMNS), dbe_temp_path, True)
if invalid_first_write:
    append_csv_chunk(
        pd.DataFrame(columns=[*STAGING_COLUMNS, "record_exclusion_reason"]),
        invalid_temp_path,
        True,
    )
os.replace(dbe_temp_path, DBE_AUDIT_PATH)
os.replace(invalid_temp_path, INVALID_RECORDS_PATH)

staging_summary = pd.DataFrame(staging_counts.items(), columns=["measure", "rows"])
timestamp_status_table = pd.DataFrame(
    staging_timestamp_status.items(), columns=["timestamp_alignment_status", "rows"]
)
partition_paths = sorted(PARTITION_DIR.glob("partition_*.csv"))
display(staging_summary)
display(timestamp_status_table.sort_values("rows", ascending=False))
print(f"Household partitions written: {len(partition_paths)}")
if len(partition_paths) != DUPLICATE_PARTITION_COUNT:
    raise RuntimeError(
        f"Expected {DUPLICATE_PARTITION_COUNT} populated partitions, found {len(partition_paths)}"
    )

## 7. Complete duplicate resolution, time-valid enrichment, and final flags

Exact duplicate groups retain one deterministic representative. Conflicting
groups are exported in full and their retained representative is excluded from
daily aggregation. High-value flags are EDA annotations only and never removal
conditions. Each processed partition is deleted immediately after being safely
appended, keeping the CSV workflow within bounded disk space.

In [ ]:
DUPLICATE_KEY_COLUMNS = [
    "household_id", "measurement_timestamp_local", "obis_normalized"
]
DUPLICATE_COMPARE_COLUMNS = NUMERIC_COLUMNS
STAGING_DTYPE_MAP = {
    **{column: "float64" for column in NUMERIC_COLUMNS},
    "household_id": "string", "measurement_time": "string",
    "time_slot_index": "Int16", "obis_normalized": "string",
    "has_invalid_numeric_text": "boolean",
    "timestamp_difference_minutes": "float64",
    "timestamp_alignment_status": "string", "is_off_15min_grid": "boolean",
    "is_missing_import_demand": "boolean",
    "is_negative_import_demand": "boolean", "is_zero_import": "boolean",
    "source_file": "string", "source_row_number": "Int64",
}
STAGING_DATE_COLUMNS = ["measurement_timestamp_local", "measurement_date"]

conflict_temp_path = temp_csv_path(DUPLICATE_CONFLICTS_PATH)
conflict_first_write = True
primary_first_write = True
day_coverage_frames = []
duplicate_counts = Counter()
final_counts = Counter()
metadata_match_counts = Counter()
final_numeric_missing_before = Counter()
final_output_max_import = float("-inf")
conflict_max_import = float("-inf")
post_resolution_duplicate_keys = 0

for partition_number, partition_path in enumerate(partition_paths):
    partition = read_dataset_csv(
        partition_path,
        dtype_map=STAGING_DTYPE_MAP,
        date_columns=STAGING_DATE_COLUMNS,
    )
    partition = partition.sort_values(
        [*DUPLICATE_KEY_COLUMNS, "source_file", "source_row_number"],
        kind="mergesort",
    ).reset_index(drop=True)
    duplicate_mask = partition.duplicated(DUPLICATE_KEY_COLUMNS, keep=False)
    partition["duplicate_status"] = pd.Series(
        "unique", index=partition.index, dtype="string"
    )

    if duplicate_mask.any():
        duplicate_rows = partition.loc[duplicate_mask].copy()
        group_sizes = duplicate_rows.groupby(
            DUPLICATE_KEY_COLUMNS, sort=False, dropna=False
        ).size()
        variation = duplicate_rows.groupby(
            DUPLICATE_KEY_COLUMNS, sort=False, dropna=False
        )[DUPLICATE_COMPARE_COLUMNS].nunique(dropna=False).max(axis=1)
        exact_keys = variation.index[variation.le(1)]
        conflict_keys = variation.index[variation.gt(1)]
        partition_key_index = pd.MultiIndex.from_frame(
            partition[DUPLICATE_KEY_COLUMNS]
        )
        exact_mask = partition_key_index.isin(exact_keys)
        conflict_mask = partition_key_index.isin(conflict_keys)
        partition.loc[exact_mask, "duplicate_status"] = "exact_duplicate_group"
        partition.loc[conflict_mask, "duplicate_status"] = "conflicting_duplicate_group"

        duplicate_counts["duplicate_rows"] += len(duplicate_rows)
        duplicate_counts["duplicate_groups"] += len(group_sizes)
        duplicate_counts["exact_duplicate_groups"] += len(exact_keys)
        duplicate_counts["conflicting_duplicate_groups"] += len(conflict_keys)
        duplicate_counts["duplicate_extra_rows_removed"] += int((group_sizes - 1).sum())

        conflicting_originals = partition.loc[conflict_mask].copy()
        if not conflicting_originals.empty:
            if conflicting_originals["avg_import_kw"].notna().any():
                conflict_max_import = max(
                    conflict_max_import,
                    float(conflicting_originals["avg_import_kw"].max()),
                )
            append_csv_chunk(
                conflicting_originals,
                conflict_temp_path,
                conflict_first_write,
            )
            conflict_first_write = False

    resolved = partition.drop_duplicates(
        DUPLICATE_KEY_COLUMNS, keep="first"
    ).copy()
    resolved["is_duplicate_conflict"] = resolved["duplicate_status"].eq(
        "conflicting_duplicate_group"
    )
    post_resolution_duplicate_keys += int(
        resolved.duplicated(DUPLICATE_KEY_COLUMNS, keep=False).sum()
    )

    partition_households = set(resolved["household_id"].astype(str).unique())
    partition_metadata = metadata_events[
        metadata_events["household_id"].astype(str).isin(partition_households)
    ].sort_values(["metadata_valid_from", "household_id"], kind="mergesort")
    resolved = resolved.sort_values(
        ["measurement_timestamp_local", "household_id"], kind="mergesort"
    )
    enriched = pd.merge_asof(
        resolved,
        partition_metadata,
        left_on="measurement_timestamp_local",
        right_on="metadata_valid_from",
        by="household_id",
        direction="backward",
        allow_exact_matches=True,
    )
    enriched["network_metadata_match_type"] = np.where(
        enriched["metadata_valid_from"].notna(),
        "backward_valid",
        "unknown_before_first_metadata",
    )
    for column in NETWORK_METADATA_COLUMNS:
        enriched[column] = enriched[column].astype("string").fillna("Unknown")
    enriched = enriched.drop(columns="metadata_valid_from")
    enriched["network_metadata_match_type"] = enriched[
        "network_metadata_match_type"
    ].astype("string")
    metadata_match_counts.update(
        enriched["network_metadata_match_type"].astype(str)
    )

    enriched["is_global_iqr_extreme_eda_only"] = enriched[
        "avg_import_kw"
    ].gt(global_iqr_extreme_limit).fillna(False).astype(bool)
    enriched["is_global_top_1pct_eda_only"] = enriched[
        "avg_import_kw"
    ].gt(global_top_1pct_limit).fillna(False).astype(bool)
    enriched["is_global_top_0_1pct_eda_only"] = enriched[
        "avg_import_kw"
    ].gt(global_top_0_1pct_limit).fillna(False).astype(bool)
    enriched["is_valid_for_daily_aggregation"] = (
        enriched["avg_import_kw"].notna()
        & enriched["avg_import_kw"].ge(0)
        & ~enriched["is_duplicate_conflict"].fillna(False)
        & ~enriched["has_invalid_numeric_text"].fillna(False)
    ).astype(bool)
    enriched["interval_exclusion_reason"] = np.select(
        [
            enriched["is_duplicate_conflict"].fillna(False).to_numpy(dtype=bool),
            enriched["has_invalid_numeric_text"].fillna(False).to_numpy(dtype=bool),
            enriched["is_missing_import_demand"].fillna(False).to_numpy(dtype=bool),
            enriched["is_negative_import_demand"].fillna(False).to_numpy(dtype=bool),
        ],
        [
            "conflicting_duplicate", "invalid_numeric_text",
            "missing_import_demand", "negative_import_demand",
        ],
        default="none",
    ).astype(str)
    enriched["quality_status"] = np.where(
        enriched["is_valid_for_daily_aggregation"],
        "valid_for_daily_aggregation",
        "excluded_from_daily_aggregation",
    ).astype(str)
    enriched["is_imputed"] = False

    final_counts["primary_rows"] += len(enriched)
    final_counts["valid_aggregation_rows"] += int(
        enriched["is_valid_for_daily_aggregation"].sum()
    )
    final_counts["missing_import_rows"] += int(
        enriched["is_missing_import_demand"].sum()
    )
    final_counts["negative_import_rows"] += int(
        enriched["is_negative_import_demand"].sum()
    )
    final_counts["off_grid_rows"] += int(enriched["is_off_15min_grid"].sum())
    final_counts["zero_import_rows"] += int(enriched["is_zero_import"].sum())
    final_counts["global_iqr_extreme_rows_retained"] += int(
        enriched["is_global_iqr_extreme_eda_only"].sum()
    )
    final_counts["global_top_1pct_rows_retained"] += int(
        enriched["is_global_top_1pct_eda_only"].sum()
    )
    final_counts["global_top_0_1pct_rows_retained"] += int(
        enriched["is_global_top_0_1pct_eda_only"].sum()
    )
    final_counts["conflicting_representative_rows"] += int(
        enriched["is_duplicate_conflict"].sum()
    )
    if enriched["avg_import_kw"].notna().any():
        final_output_max_import = max(
            final_output_max_import, float(enriched["avg_import_kw"].max())
        )

    day_coverage = enriched.groupby(
        ["household_id", "measurement_date"], sort=False
    ).agg(
        raw_lp_rows=("household_id", "size"),
        unique_slot_count=("time_slot_index", "nunique"),
        valid_interval_count=("is_valid_for_daily_aggregation", "sum"),
        missing_import_count=("is_missing_import_demand", "sum"),
        duplicate_conflict_count=("is_duplicate_conflict", "sum"),
        statistical_extreme_count=("is_global_iqr_extreme_eda_only", "sum"),
    ).reset_index()
    day_coverage["coverage_ratio"] = (
        day_coverage["valid_interval_count"] / EXPECTED_INTERVALS_PER_DAY
    )
    day_coverage["coverage_status"] = np.select(
        [
            day_coverage["duplicate_conflict_count"].gt(0).to_numpy(dtype=bool),
            day_coverage["valid_interval_count"].eq(EXPECTED_INTERVALS_PER_DAY).to_numpy(dtype=bool),
            day_coverage["valid_interval_count"].ge(DESCRIPTIVE_NEAR_COMPLETE_MIN_INTERVALS).to_numpy(dtype=bool),
        ],
        ["invalid_duplicate_conflict", "complete_96", "near_complete_70_95"],
        default="severely_incomplete_below_70",
    ).astype(str)
    day_coverage["is_complete_96_day"] = (
        day_coverage["valid_interval_count"].eq(EXPECTED_INTERVALS_PER_DAY)
        & day_coverage["duplicate_conflict_count"].eq(0)
        & day_coverage["unique_slot_count"].eq(EXPECTED_INTERVALS_PER_DAY)
    )
    for minimum_intervals in (95, 94, 93, 92):
        day_coverage[f"is_{minimum_intervals}_plus_day"] = day_coverage["valid_interval_count"].ge(minimum_intervals)
    # Temporary compatibility alias: descriptive 96/96 completeness only, not final target eligibility.
    day_coverage["is_valid_primary_target_day"] = day_coverage["is_complete_96_day"]
    day_coverage_frames.append(day_coverage)

    enriched = enriched.sort_values(
        ["household_id", "measurement_timestamp_local"], kind="mergesort"
    )
    output_frame = enriched[PRIMARY_OUTPUT_COLUMNS]
    for column in PRIMARY_MEASUREMENT_COLUMNS + ["timestamp_difference_minutes"]:
        final_numeric_missing_before[column] += int(output_frame[column].isna().sum())
    append_csv_chunk(output_frame, PRIMARY_TEMP_PATH, primary_first_write)
    primary_first_write = False

    partition_path.unlink()
    if (partition_number + 1) % 8 == 0:
        print(f"Resolved {partition_number + 1} household partitions")

    del partition, resolved, enriched, output_frame, day_coverage
    gc.collect()

if conflict_first_write:
    append_csv_chunk(
        pd.DataFrame(columns=[*STAGING_COLUMNS, "duplicate_status"]),
        conflict_temp_path,
        True,
    )
os.replace(conflict_temp_path, DUPLICATE_CONFLICTS_PATH)
if post_resolution_duplicate_keys:
    raise RuntimeError(
        f"Duplicate keys remain after resolution: {post_resolution_duplicate_keys}"
    )
if primary_first_write or not PRIMARY_TEMP_PATH.exists():
    raise RuntimeError("Primary temporary CSV was not created")
os.replace(PRIMARY_TEMP_PATH, PRIMARY_OUTPUT_PATH)

duplicate_summary = pd.DataFrame(
    duplicate_counts.items(), columns=["measure", "rows_or_groups"]
)
final_processing_summary = pd.DataFrame(
    final_counts.items(), columns=["measure", "rows"]
)
network_match_summary = pd.DataFrame(
    metadata_match_counts.items(), columns=["network_metadata_match_type", "rows"]
)
display(duplicate_summary)
display(final_processing_summary)
display(network_match_summary)

## 8. Household-day completeness audit

Completeness is based on valid unique intervals, not raw row count. A complete
96/96 day and the 95/94/93/92-plus flags are descriptive coverage indicators.
The 70-interval boundary separates near-complete from lower-coverage days for
audit summaries only. The legacy `is_valid_primary_target_day` field is retained for
current downstream schema compatibility and mirrors `is_complete_96_day`; it is not
a target-eligibility decision. This notebook does not decide target-day or history
eligibility; those policies belong to the downstream target-design/modeling stages.

In [ ]:
household_day_coverage = pd.concat(day_coverage_frames, ignore_index=True)
household_day_coverage = household_day_coverage.sort_values(
    ["household_id", "measurement_date"], kind="mergesort"
).reset_index(drop=True)
if household_day_coverage.duplicated(
    ["household_id", "measurement_date"]
).any():
    raise RuntimeError("Coverage keys are not unique before writing")
write_csv_atomic(household_day_coverage, DAY_COVERAGE_PATH)

final_coverage_summary = household_day_coverage[
    "coverage_status"
].value_counts().rename_axis("coverage_status").reset_index(name="household_days")
final_valid_interval_distribution = household_day_coverage[
    "valid_interval_count"
].value_counts().sort_index().rename_axis("valid_interval_count").reset_index(
    name="household_days"
)
display(final_coverage_summary)
display(
    household_day_coverage[
        [
            "raw_lp_rows", "unique_slot_count", "valid_interval_count",
            "missing_import_count", "duplicate_conflict_count", "coverage_ratio",
        ]
    ].describe().T
)
print(f"Household-days audited: {len(household_day_coverage):,}")
print(
    "Fully complete observed household-days (96/96; descriptive): "
    f"{int(household_day_coverage['is_complete_96_day'].sum()):,}"
)

## 9. Full CSV round-trip validation

The final file is independently re-read with the declared datatype contract.
Validation covers row reconciliation, ordering, duplicate keys, datetime parsing,
booleans, numerical missingness, retained maximum demand, absence of imputation,
coverage-key uniqueness, and audit-file readability.

In [ ]:
primary_header = list(pd.read_csv(PRIMARY_OUTPUT_PATH, nrows=0).columns)
missing_required_columns = sorted(set(PRIMARY_OUTPUT_COLUMNS) - set(primary_header))
if missing_required_columns or primary_header != PRIMARY_OUTPUT_COLUMNS:
    raise RuntimeError(
        f"Primary CSV schema mismatch; missing={missing_required_columns}"
    )

validation_counts = Counter()
validation_obis = Counter()
validation_metadata = Counter()
validation_exclusion_reasons = Counter()
validation_numeric_missing = Counter()
validation_households = set()
validation_max_import = float("-inf")
validation_duplicate_keys = 0
validation_datetime_failures = 0
ordering_errors = 0
previous_last_key = None

for frame in read_dataset_csv(
    PRIMARY_OUTPUT_PATH,
    dtype_map=PRIMARY_DTYPE_MAP,
    date_columns=PRIMARY_DATE_COLUMNS,
    chunksize=CHUNK_SIZE,
):
    validation_counts["rows"] += len(frame)
    validation_households.update(frame["household_id"].dropna().astype(str).unique())
    validation_counts["missing_import_rows"] += int(frame["avg_import_kw"].isna().sum())
    validation_counts["valid_aggregation_rows"] += int(
        frame["is_valid_for_daily_aggregation"].fillna(False).sum()
    )
    validation_counts["duplicate_conflict_rows"] += int(
        frame["is_duplicate_conflict"].fillna(False).sum()
    )
    validation_counts["statistical_extreme_rows_retained"] += int(
        frame["is_global_iqr_extreme_eda_only"].fillna(False).sum()
    )
    validation_counts["top_1pct_rows_retained"] += int(
        frame["is_global_top_1pct_eda_only"].fillna(False).sum()
    )
    validation_counts["top_0_1pct_rows_retained"] += int(
        frame["is_global_top_0_1pct_eda_only"].fillna(False).sum()
    )
    validation_counts["imputed_rows"] += int(frame["is_imputed"].fillna(False).sum())
    validation_obis.update(frame["obis_normalized"].astype(str))
    validation_metadata.update(frame["network_metadata_match_type"].astype(str))
    validation_exclusion_reasons.update(frame["interval_exclusion_reason"].astype(str))
    for column in PRIMARY_MEASUREMENT_COLUMNS + ["timestamp_difference_minutes"]:
        validation_numeric_missing[column] += int(frame[column].isna().sum())
    validation_datetime_failures += int(
        frame[PRIMARY_DATE_COLUMNS].isna().any(axis=1).sum()
    )
    if frame["avg_import_kw"].notna().any():
        validation_max_import = max(
            validation_max_import, float(frame["avg_import_kw"].max())
        )

    key_frame = frame[
        ["household_id", "measurement_timestamp_local", "obis_normalized"]
    ]
    validation_duplicate_keys += int(key_frame.duplicated(keep=False).sum())
    order_index = pd.MultiIndex.from_frame(
        frame[["household_id", "measurement_timestamp_local"]]
    )
    if not order_index.is_monotonic_increasing:
        ordering_errors += 1
    first_key = (
        str(frame.iloc[0]["household_id"]),
        frame.iloc[0]["measurement_timestamp_local"],
    )
    last_key = (
        str(frame.iloc[-1]["household_id"]),
        frame.iloc[-1]["measurement_timestamp_local"],
    )
    if previous_last_key is not None and first_key < previous_last_key:
        ordering_errors += 1
    if previous_last_key is not None and first_key == previous_last_key:
        validation_duplicate_keys += 1
    previous_last_key = last_key

expected_primary_rows = int(
    staging_counts["valid_key_lp_rows"]
    - duplicate_counts["duplicate_extra_rows_removed"]
)
primary_rows = int(validation_counts["rows"])

coverage_roundtrip = read_dataset_csv(
    DAY_COVERAGE_PATH,
    dtype_map={
        "household_id": "string", "raw_lp_rows": "Int64",
        "unique_slot_count": "Int64", "valid_interval_count": "Int64",
        "missing_import_count": "Int64", "duplicate_conflict_count": "Int64",
        "statistical_extreme_count": "Int64", "coverage_ratio": "float64",
        "coverage_status": "string", "is_complete_96_day": "boolean",
        "is_95_plus_day": "boolean", "is_94_plus_day": "boolean",
        "is_93_plus_day": "boolean", "is_92_plus_day": "boolean",
        "is_valid_primary_target_day": "boolean",
    },
    date_columns=["measurement_date"],
)
coverage_duplicate_keys = int(
    coverage_roundtrip.duplicated(["household_id", "measurement_date"]).sum()
)

supporting_headers_readable = all(
    len(pd.read_csv(path, nrows=0).columns) > 0
    for path in [DBE_AUDIT_PATH, INVALID_RECORDS_PATH, DUPLICATE_CONFLICTS_PATH]
)
output_dataset_paths = [
    PRIMARY_OUTPUT_PATH, DBE_AUDIT_PATH, INVALID_RECORDS_PATH,
    DUPLICATE_CONFLICTS_PATH, DAY_COVERAGE_PATH,
]
row_reconciliation = pd.DataFrame(
    {
        "measure": [
            "merged_input_rows", "discovered_lp_rows", "staged_valid_key_lp_rows",
            "invalid_key_lp_rows", "dbe_rows", "unexpected_obis_rows",
            "duplicate_extra_rows_removed", "expected_primary_rows",
            "actual_primary_rows",
        ],
        "value": [
            discovery_total_rows, discovery_obis_counts.get("LP", 0),
            staging_counts["valid_key_lp_rows"], staging_counts["invalid_key_lp_rows"],
            staging_counts["dbe_rows"], staging_counts["unexpected_obis_rows"],
            duplicate_counts["duplicate_extra_rows_removed"], expected_primary_rows,
            primary_rows,
        ],
    }
)
validation_checks = {
    "all_dataset_outputs_are_csv": all(path.suffix.lower() == ".csv" for path in output_dataset_paths),
    "primary_schema_matches_contract": primary_header == PRIMARY_OUTPUT_COLUMNS,
    "row_reconciliation_matches": primary_rows == expected_primary_rows,
    "only_lp_records": set(validation_obis) == {PRIMARY_OBIS},
    "duplicate_keys_resolved": validation_duplicate_keys == 0 and post_resolution_duplicate_keys == 0,
    "globally_sorted_by_household_timestamp": ordering_errors == 0,
    "datetime_columns_reparse": validation_datetime_failures == 0,
    "numeric_missingness_roundtrip_matches": dict(validation_numeric_missing) == dict(final_numeric_missing_before),
    "no_future_metadata_matches": not any("forward" in key for key in validation_metadata),
    "missing_flag_matches_values": validation_counts["missing_import_rows"] == final_counts["missing_import_rows"],
    "no_interval_imputation": validation_counts["imputed_rows"] == 0,
    "statistical_extremes_retained": validation_counts["statistical_extreme_rows_retained"] == final_counts["global_iqr_extreme_rows_retained"],
    "maximum_demand_43_2648_retained": np.isclose(validation_max_import, 43.2648, atol=1e-12) and np.isclose(discovery_lp_max_import, 43.2648, atol=1e-12),
    "coverage_keys_unique": coverage_duplicate_keys == 0,
    "coverage_rows_reconcile": len(coverage_roundtrip) == len(household_day_coverage),
    "supporting_csv_headers_readable": supporting_headers_readable,
    "expected_intervals_derived_from_data": EXPECTED_INTERVALS_PER_DAY == derived_expected_intervals_per_day == 96,
    "descriptive_near_complete_boundary_recorded": DESCRIPTIVE_NEAR_COMPLETE_MIN_INTERVALS == 70,
    "coverage_flags_match_descriptive_definitions": (
        household_day_coverage["is_complete_96_day"].eq(
            household_day_coverage["valid_interval_count"].eq(EXPECTED_INTERVALS_PER_DAY)
            & household_day_coverage["unique_slot_count"].eq(EXPECTED_INTERVALS_PER_DAY)
            & household_day_coverage["duplicate_conflict_count"].eq(0)
        ).all()
        and household_day_coverage["is_valid_primary_target_day"].eq(
            household_day_coverage["is_complete_96_day"]
        ).all()
        and all(
            household_day_coverage[f"is_{threshold}_plus_day"].eq(
                household_day_coverage["valid_interval_count"].ge(threshold)
            ).all()
            for threshold in (95, 94, 93, 92)
        )
    ),
}
validation_table = pd.DataFrame(
    {"check": list(validation_checks), "passed": list(validation_checks.values())}
)
display(row_reconciliation)
display(validation_table)
display(pd.DataFrame(validation_obis.items(), columns=["obis", "rows"]))
display(pd.DataFrame(validation_metadata.items(), columns=["network_metadata_match_type", "rows"]))
display(pd.DataFrame(validation_exclusion_reasons.items(), columns=["interval_exclusion_reason", "rows"]))

failed_checks = validation_table.loc[~validation_table["passed"], "check"].tolist()
if failed_checks:
    raise RuntimeError(f"Preprocessing validation failed: {failed_checks}")

if WORK_DIR.exists():
    ensure_inside_project(WORK_DIR)
    shutil.rmtree(WORK_DIR)
temporary_partitions_removed = not WORK_DIR.exists()
if not temporary_partitions_removed:
    raise RuntimeError("Temporary CSV partitions were not removed")

## 10. Reports, datatype contract, manifest, and completion gate

In [ ]:
preprocessing_summary = pd.DataFrame(
    {
        "measure": [
            "input_rows", "input_households", "input_lp_rows", "input_dbe_rows",
            "primary_output_rows", "primary_output_households",
            "valid_aggregation_rows", "missing_import_rows_retained",
            "statistical_extreme_rows_retained", "exact_duplicate_groups",
            "conflicting_duplicate_groups", "duplicate_extra_rows_removed",
            "expected_intervals_per_day", "descriptive_near_complete_min_intervals",
            "complete_household_days", "near_complete_household_days",
            "severely_incomplete_household_days",
        ],
        "value": [
            discovery_total_rows, len(discovery_households),
            discovery_obis_counts.get("LP", 0), discovery_obis_counts.get("DBE", 0),
            primary_rows, len(validation_households),
            validation_counts["valid_aggregation_rows"],
            validation_counts["missing_import_rows"],
            validation_counts["statistical_extreme_rows_retained"],
            duplicate_counts["exact_duplicate_groups"],
            duplicate_counts["conflicting_duplicate_groups"],
            duplicate_counts["duplicate_extra_rows_removed"],
            EXPECTED_INTERVALS_PER_DAY, DESCRIPTIVE_NEAR_COMPLETE_MIN_INTERVALS,
            int((household_day_coverage["coverage_status"] == "complete_96").sum()),
            int((household_day_coverage["coverage_status"] == "near_complete_70_95").sum()),
            int((household_day_coverage["coverage_status"] == "severely_incomplete_below_70").sum()),
        ],
    }
)
missingness_summary = pd.DataFrame(
    {
        "measure": [
            "lp_missing_avg_import_kw_before_duplicate_resolution",
            "primary_missing_avg_import_kw_after_duplicate_resolution",
            "primary_zero_avg_import_kw", "invalid_numeric_text_rows",
        ],
        "rows": [
            discovery_lp_missing_import, validation_counts["missing_import_rows"],
            final_counts["zero_import_rows"], staging_invalid_numeric_rows,
        ],
    }
)
schema_contract_table = pd.DataFrame(
    {
        "column": PRIMARY_OUTPUT_COLUMNS,
        "logical_dtype": [
            "datetime" if column in PRIMARY_DATE_COLUMNS
            else str(PRIMARY_DTYPE_MAP.get(column, "string"))
            for column in PRIMARY_OUTPUT_COLUMNS
        ],
    }
)

report_tables = {
    "01_environment.csv": environment_table,
    "01_input_contract_checks.csv": input_contract_checks,
    "01_preprocessing_summary.csv": preprocessing_summary,
    "01_cadence_evidence.csv": cadence_evidence,
    "01_preliminary_coverage_summary.csv": preliminary_coverage_summary,
    "01_final_coverage_summary.csv": final_coverage_summary,
    "01_valid_interval_distribution.csv": final_valid_interval_distribution,
    "01_timestamp_evidence.csv": timestamp_alignment_evidence,
    "01_timestamp_status_summary.csv": timestamp_status_table,
    "01_duplicate_summary.csv": duplicate_summary,
    "01_missingness_summary.csv": missingness_summary,
    "01_network_metadata_summary.csv": network_metadata_audit,
    "01_network_match_summary.csv": network_match_summary,
    "01_changing_metadata_detail.csv": changing_metadata_detail,
    "01_statistical_extreme_reference.csv": extreme_reference_table,
    "01_row_reconciliation.csv": row_reconciliation,
    "01_validation_checks.csv": validation_table,
    "01_primary_csv_schema_contract.csv": schema_contract_table,
}
for filename, frame in report_tables.items():
    write_csv_atomic(frame, REPORT_TABLE_DIR / filename)

dtype_contract = {
    "contract_version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "path": str(PRIMARY_OUTPUT_PATH.relative_to(PROJECT_ROOT)),
    "encoding": CSV_ENCODING,
    "separator": CSV_SEPARATOR,
    "missing_value": CSV_MISSING_VALUE,
    "date_columns": PRIMARY_DATE_COLUMNS,
    "columns": {
        row.column: row.logical_dtype
        for row in schema_contract_table.itertuples(index=False)
    },
    "reader": {
        "true_values": ["True"], "false_values": ["False"],
        "keep_default_na": False, "na_values": [""],
    },
}
write_json_atomic(dtype_contract, DTYPE_CONTRACT_PATH)

primary_signature = {
    "path": str(PRIMARY_OUTPUT_PATH.relative_to(PROJECT_ROOT)),
    "format": "csv", "rows": primary_rows,
    "columns": len(PRIMARY_OUTPUT_COLUMNS),
    "size_bytes": PRIMARY_OUTPUT_PATH.stat().st_size,
    "sha256": sha256_file(PRIMARY_OUTPUT_PATH),
}
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "research_stage": "interval_preprocessing",
    "execution_mode": EXECUTION_MODE,
    "primary_output": str(PRIMARY_OUTPUT_PATH.relative_to(PROJECT_ROOT)),
    "primary_rows": primary_rows,
    "primary_columns": PRIMARY_OUTPUT_COLUMNS,
    "primary_signature": primary_signature,
    "dtype_contract": str(DTYPE_CONTRACT_PATH.relative_to(PROJECT_ROOT)),
    "input": {
        "merged_15min_path": str(RAW_MERGED_DATA_PATH.relative_to(PROJECT_ROOT)),
        "merged_15min_rows": discovery_total_rows,
        "merged_6hour_metadata_path": str(SIX_HOUR_MERGED_DATA_PATH.relative_to(PROJECT_ROOT)),
        "source_row_counts": dict(discovery_source_counts),
    },
    "policy": {
        "timezone": TIMEZONE,
        "canonical_timestamp_source": CANONICAL_TIMESTAMP_SOURCE,
        "primary_obis": PRIMARY_OBIS,
        "observed_interval_minutes": observed_interval_minutes,
        "expected_intervals_per_day": EXPECTED_INTERVALS_PER_DAY,
        "descriptive_near_complete_min_intervals": DESCRIPTIVE_NEAR_COMPLETE_MIN_INTERVALS,
        "day_completeness_role": "descriptive; target and history eligibility selected downstream",
        "complete_96_flag": "is_complete_96_day; observed completeness only",
        "coverage_threshold_flags": "is_95_plus_day, is_94_plus_day, is_93_plus_day, is_92_plus_day; descriptive only",
        "compatibility_alias": "is_valid_primary_target_day mirrors is_complete_96_day temporarily; not final eligibility",
        "missing_import_imputation": "not_applied",
        "statistical_extremes_removed": 0,
        "future_metadata_fill_allowed": False,
        "metadata_join": "backward_asof_by_household",
    },
    "counts": {
        **{key: int(value) for key, value in staging_counts.items()},
        **{key: int(value) for key, value in duplicate_counts.items()},
        **{key: int(value) for key, value in final_counts.items()},
    },
    "supporting_outputs": {
        "dbe_records": str(DBE_AUDIT_PATH.relative_to(PROJECT_ROOT)),
        "invalid_key_records": str(INVALID_RECORDS_PATH.relative_to(PROJECT_ROOT)),
        "duplicate_conflicts": str(DUPLICATE_CONFLICTS_PATH.relative_to(PROJECT_ROOT)),
        "household_day_coverage_audit": str(DAY_COVERAGE_PATH.relative_to(PROJECT_ROOT)),
    },
    "validation": {
        **{key: bool(value) for key, value in validation_checks.items()},
        "temporary_csv_partitions_removed": temporary_partitions_removed,
    },
    "software": {
        "python": platform.python_version(), "pandas": pd.__version__,
        "numpy": np.__version__,
    },
}
write_json_atomic(manifest, MANIFEST_PATH)

PREPROCESSING_COMPLETE = all(manifest["validation"].values())
if not PREPROCESSING_COMPLETE:
    raise RuntimeError("Preprocessing did not pass the completion gate")

display(preprocessing_summary)
print("PREPROCESSING COMPLETE")
print(f"Primary reusable CSV: {PRIMARY_OUTPUT_PATH}")
print(f"Rows: {primary_rows:,}")
print(f"Valid daily-aggregation intervals: {validation_counts['valid_aggregation_rows']:,}")
print(f"Missing demand rows retained: {validation_counts['missing_import_rows']:,}")
print(f"Statistical extremes retained: {validation_counts['statistical_extreme_rows_retained']:,}")
print(f"Expected intervals/day: {EXPECTED_INTERVALS_PER_DAY}")
print(f"Descriptive near-complete minimum intervals: {DESCRIPTIVE_NEAR_COMPLETE_MIN_INTERVALS}")
print(f"Manifest: {MANIFEST_PATH}")

## Completion interpretation

A successful run means the interval CSV is reproducible, globally ordered, and
suitable for streaming EDA and household-day construction. It remains an interval
dataset—not a model-training dataset. Missing demand stays missing and high demand is
retained for research analysis. Household-day coverage fields describe data quality;
target-day and historical-feature eligibility are intentionally decided downstream.